# 🤖 QA Agent Notebook
Este notebook genera matrices de pruebas y casos Gherkin usando OpenAI API.
- Configura tu `.env` con la API Key.
- Ejecuta celda por celda.


In [ ]:
!pip install "openai>=1.0.0" pandas openpyxl python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

load_dotenv()
client = OpenAI()  # Lee OPENAI_API_KEY desde .env

In [ ]:
requerimiento = """
Titulo:[IOS] Generar acceso en menú hamburguesa con la etiqueta nuevo (estrategia de liberación)

Objetivo: Implementar lógica desagrupada por segmento (iniciando con premier) 
y generar acceso en el menú con flag de LaunchDarkly.

Bancas: PBP, P, PBM, PE, PP, PR, PBU, PRE

Criterios de aceptación:
- Segmentar clientes según bancas y excluir/incluir casos específicos.
- Incorporar flag LaunchDarkly apagado por defecto, liberación progresiva.
- El acceso debe estar en 3ra posición del menú con ícono: Outlined/ic_dialog_group_outlined.
"""

In [ ]:
prompt = f"""
Eres un QA Engineer experto en generar matrices de prueba y casos Gherkin (Cucumber).
Aquí tienes un ejemplo de cómo debe ser la salida:

---
Ejemplo de matriz de prueba:

| ID | Escenario | Datos de entrada | Resultado esperado |
|----|-----------|------------------|--------------------|
| TC001 | Usuario con acceso válido | Cliente segmentado en banca Premier | El acceso aparece en el menú hamburguesa |
| TC002 | Usuario sin acceso | Cliente con cuenta digital | El acceso NO debe mostrarse |
| TC003 | Validación de flag LaunchDarkly | Flag apagado | El acceso NO aparece |
| TC004 | Validación de flag LaunchDarkly | Flag encendido | El acceso aparece en el menú |
| TC005 | Posición en menú | Cliente Premier | El acceso aparece en la tercera posición con el ícono correcto |

Ejemplo de casos Gherkin:

Feature: Validar acceso en menú hamburguesa para banca Premier

  Scenario: Cliente Premier con flag apagado
    Given un cliente segmentado en banca Premier
    When el flag LaunchDarkly está apagado
    Then el acceso no debe aparecer en el menú

  Scenario: Cliente Premier con flag encendido
    Given un cliente segmentado en banca Premier
    When el flag LaunchDarkly está encendido
    Then el acceso debe aparecer en la tercera posición del menú con el ícono Outlined/ic_dialog_group_outlined
---

Ahora, genera la **matriz de pruebas** y los **casos Gherkin** para el siguiente requerimiento:

{requerimiento}
"""

In [ ]:
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.3
)

output = resp.choices[0].message.content
print("=== SALIDA IA ===")
print(output)

In [ ]:
rows = []
for line in output.splitlines():
    if "|" in line and "---" not in line:
        cols = [c.strip() for c in line.split("|") if c.strip()]
        if len(cols) > 1:
            rows.append(cols)

if rows:
    df = pd.DataFrame(rows[1:], columns=rows[0])
    df.to_excel("matriz_pruebas.xlsx", index=False)
    print("✅ matriz_pruebas.xlsx generado")

gherkin = []
capture = False
for line in output.splitlines():
    if line.strip().startswith(("Feature", "Scenario")):
        capture = True
    if capture:
        gherkin.append(line)

if gherkin:
    with open("casos.feature", "w", encoding="utf-8") as f:
        f.write("\n".join(gherkin))
    print("✅ casos.feature generado")